## Lab Objective {.unnumbered}

In this lab, you will:

- Use **Google Colab in the browser**
- Load real-world corporate text data (SEC 10-K filings)
- Implement a **classical NLP preprocessing pipeline**
- Answer **exploratory questions** about corporate disclosures using text analytics

This lab establishes the **computational and conceptual foundation** for later work with embeddings and generative models.


## Background Context {.unnumbered}

Public companies file **Form 10-K** annually with the U.S. Securities and Exchange Commission (SEC).  
These filings contain rich textual information about:

- business operations  
- risks and uncertainties  
- management discussion  
- regulatory disclosures  

In this lab, we treat each 10-K as **raw text data** and apply a standard NLP pipeline to prepare it for analysis.


## Using Google Colab on the Web {.unnumbered}

Use **Google Colab in your browser** for this lab. This is the cleanest option for AD698 and avoids local environment setup issues.

### Open Colab {.unnumbered}

- Go to [Google Colab](https://colab.research.google.com/)
- Sign in with your Google account
- Create a new notebook or upload the notebook file for this lab

### Save to Drive {.unnumbered}

- Rename the notebook clearly
- Save it in your Google Drive so your work persists between sessions

### Runtime Guidance {.unnumbered}

- Use the default runtime for this lab
- If a later lab or assignment requires GPU, switch runtimes in Colab using `Runtime -> Change runtime type`

### Recommended Workflow {.unnumbered}

- Keep the notebook open in Colab while you run cells
- Use Google Drive for input and output files when needed
- Re-run cells from top to bottom if your session resets

## Dataset Overview {.unnumbered}

- All data for this lab is located in: [SEC-10K-2024/](https://drive.google.com/drive/folders/1q7BfsNHCewG1zNfnqyCcBj9p_RUt-zW6?usp=drive_link)
- You will need to "copy" the folder to your own Google Drive
- Right click on the folder, and then click "Add shortcut to Drive". This will allow you to access the folder from your drive!
- This folder contains **plain-text 10-K filings** for multiple publicly traded firms.
- Each file represents **one company’s annual report**.

![](./M01_lecture02_figures/gdrive-add-folder.png){width="80%" fig-align="center"}

:::{.callout-note}
When running notebooks on Google Colab, file paths such as `../data/...` will **not work**.
Colab runs on a remote virtual machine, so all data must be accessed via mounted storage or downloads.
:::


## Research Framing (Important) {.unnumbered}

You are **not** training a model yet. Instead, think of this lab as **asking structured questions of text**, such as:

- What terms dominate risk disclosures?
- How consistent is language across companies?
- Which words survive aggressive cleaning?
- How does preprocessing change the text representation?

Your answers will be supported by **intermediate outputs**, not final predictions.

## Recommended Notebook Pattern {.unnumbered}

Keep the notebook structured as a sequence of small, auditable steps:

1. Setup and data access
2. One NLP transformation at a time
3. Validation output after each transformation
4. Short interpretation of what the transformation changed

## Lab Validation Standard {.unnumbered}

After each major NLP step, show at least one of the following:

- sample text before and after processing
- token counts or sentence counts
- POS or dependency examples
- a short note explaining what the intermediate output tells you

## NLP Processing Pipeline {.unnumbered}

You will implement the following pipeline **step by step**:

1. Raw text  
2. Sentence segmentation  
3. Tokenization  
4. Part-of-Speech (POS) tagging  
5. Stop-word removal  
6. Stemming / Lemmatization  
7. Dependency parsing  
8. String metrics & matching  

```{dot}
digraph NLP {
    graph [bb="", margin=0, pad=0];
    rankdir=LR;
    splines=ortho;
    nodesep=0.5;
    ranksep=0.6;

    node [shape=box, style=rounded, fontsize=12, fontname="Roboto"];

    // Row 1
    { rank=same;
        A [label="Raw Text"];
        B [label="Sentence Segmentation"];
        C [label="Tokenization"];
        D [label="Part-of-Speech Tagging"];
    }

    // Row 2
    { rank=same;
        F [label="Stemming / Lemmatization"];
        E [label="Stop Word Removal"];
        G [label="Dependency Parsing"];
        H [label="String Metrics & Matching"];
    }

    // Edges Row 1
    A -> B -> C -> D;

    // Wrap to Row 2
    D -> E;

    // Row 2 edges
    E -> F -> G -> H;
}

```

Each stage produces **artifacts** that help you answer analytical questions.


## Load and Inspect the Data {.unnumbered}

### Mount Drive in Colab {.unnumbered}


In [ ]:
#| echo: true
#| eval: false
from google.colab import drive
drive.mount('/content/drive')


### Verify Files {.unnumbered}


In [ ]:
#| echo: true
#| eval: false
import os
os.listdir("/content/drive/MyDrive")


Adjust paths as needed.

### For Local Drive (Optional) {.unnumbered}


In [ ]:
#| echo: true
#| eval: false
from pathlib import Path

DATA_DIR = Path("../data/SEC-10K-2024")

files = list(DATA_DIR.glob("*.txt"))
print(f"Number of 10-K documents: {len(files)}")

# Read a sample document
sample_text = files[0].read_text(encoding="utf-8")
print(sample_text[:1500])


## What sections of the 10-K appear most frequently in the opening text?

This will help you understand the structure of the document and identify key areas for analysis (e.g., risk factors, management discussion). We first start with Sentence Segmentation


In [ ]:
#| echo: true
#| eval: false
import nltk
nltk.download("punkt")

from nltk.tokenize import sent_tokenize

sentences = sent_tokenize(sample_text)
print(f"Number of sentences: {len(sentences)}")
sentences[:5]


## Are sentences in 10-Ks longer or shorter than typical news or social media text?


In [ ]:
#| echo: true
#| eval: false
from nltk.tokenize import word_tokenize

tokens = word_tokenize(sample_text)
tokens[:30]


## What kinds of tokens appear that are not “words” (e.g., symbols, numbers, legal references)?


In [ ]:
#| echo: true
#| eval: false
nltk.download("averaged_perceptron_tagger")

from nltk import pos_tag

pos_tags = pos_tag(tokens[:50])
pos_tags


## Which POS categories dominate risk-related sections (nouns, verbs, adjectives)?


In [ ]:
#| echo: true
#| eval: false
from nltk.corpus import stopwords
nltk.download("stopwords")

stop_words = set(stopwords.words("english"))

filtered_tokens = [
    t.lower() for t in tokens
    if t.isalpha() and t.lower() not in stop_words
]

filtered_tokens[:30]


## Which important business terms survive stop-word removal?


In [ ]:
#| echo: true
#| eval: false
from nltk.stem import PorterStemmer, WordNetLemmatizer
nltk.download("wordnet")

stemmer = PorterStemmer()
lemmatizer = WordNetLemmatizer()

stems = [stemmer.stem(t) for t in filtered_tokens[:20]]
lemmas = [lemmatizer.lemmatize(t) for t in filtered_tokens[:20]]

list(zip(filtered_tokens[:20], stems, lemmas))


## Which transformation preserves interpretability better for financial text?


In [ ]:
#| echo: true
#| eval: false
import spacy
nlp = spacy.load("en_core_web_sm")

doc = nlp(sentences[0])
[(token.text, token.dep_, token.head.text) for token in doc]


## How might dependency relationships help identify risk statements or obligations?


In [ ]:
#| echo: true
#| eval: false
from difflib import SequenceMatcher

def similarity(a, b):
    return SequenceMatcher(None, a, b).ratio()

similarity(
    "risk management strategy",
    "enterprise risk management"
)


## Why might approximate string matching be useful for cross-company comparison?



## Deliverables

Submit word document with answering the questions in addition to the Jupyter notebook with the code and outputs (either `.ipynb` or `.pdf`):

## Key Takeaway

> Before we can generate language,
> we must first **discipline text into structure**.

This pipeline is the foundation upon which **Bag of Words, TF-IDF, embeddings, and generative models** are built.
